# Unidad 4 : Creación del agente y evaluación con GAIA

In [24]:
!pip install -q -U "smolagents[openai,toolkit]" pandas openpyxl pypdf requests
!pip install -q google-genai
from google import genai
from google.genai import types

from markdownify import markdownify
from urllib.parse import quote
from ddgs import DDGS

gclient = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

import requests, os, io, time, re
from google.colab import userdata
from smolagents import CodeAgent, DuckDuckGoSearchTool, VisitWebpageTool, OpenAIServerModel, tool

model = OpenAIServerModel(
    model_id="gemini-3.5-flash-lite",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=userdata.get("GEMINI_API_KEY"),
    max_tokens=4096,
)

Para empezar, creamos el modelo que se va a usar e importamos todas las librerias necesarias que se van a utilizar.

In [12]:
API = "https://agents-course-unit4-scoring.hf.space"

questions = requests.get(f"{API}/questions", timeout=30).json()
print(len(questions), "questions")
print(questions[0])

20 questions
{'task_id': '8e867cd7-cff9-4e6c-867a-ff5ddc2550be', 'question': 'How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest 2022 version of english wikipedia.', 'Level': '1', 'file_name': ''}


Luego, utilizando la API, recolectamos las preguntas que van a ser evaluadas.

In [26]:
os.makedirs("task_files", exist_ok=True)

@tool
def read_task_file(task_id: str) -> str:
    """Downloads the file attached to a task and returns its contents (text, CSV, Excel, PDF, code) or just its saved path for other types.
    Args:
        task_id: The task_id of the question that has an attached file.
    """
    r = requests.get(f"{API}/files/{task_id}", timeout=60)
    if r.status_code != 200:
        return f"No file available (status {r.status_code})."
    cd = r.headers.get("content-disposition", "")
    m = re.search(r'filename="?([^";]+)"?', cd)
    name = m.group(1) if m else task_id
    path = os.path.join("task_files", name)
    with open(path, "wb") as f:
        f.write(r.content)
    ext = name.lower().rsplit(".", 1)[-1]
    try:
        if ext in ("txt", "csv", "json", "md", "py", "html", "xml", "jsonl"):
            return f"Saved to {path}.\n" + r.content.decode("utf-8", errors="replace")[:20000]
        if ext in ("xlsx", "xls"):
            import pandas as pd
            sheets = pd.read_excel(path, sheet_name=None)
            return f"Saved to {path}.\n" + "\n\n".join(f"Sheet {k}:\n{v.to_string()}" for k, v in sheets.items())[:20000]
        if ext == "pdf":
            from pypdf import PdfReader
            text = "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
            return f"Saved to {path}.\n" + text[:20000]
    except Exception as e:
        return f"Saved to {path}, but reading failed: {e}"
    return f"Saved to {path}. This file type ({ext}) can't be read directly."



UA = {"User-Agent": "GAIAAgentCourse/1.0 (https://huggingface.co/spaces/LisSanchez-DS/Unit4_GAIA_Agent; student project)"}

_last_search = [0.0]

@tool
def web_search(query: str) -> str:
    """Searches the web with DuckDuckGo and returns the top results with titles, links, and snippets. Use plain keywords, no 'site:' operators.

    Args:
        query: The search query.
    """
    query = re.sub(r"site:\S+", "", query).strip()
    for attempt in range(2):
        try:
            results = DDGS(timeout=8).text(query, max_results=8, backend="duckduckgo")
            if results:
                return "\n\n".join(f"{r['title']}\n{r['href']}\n{r['body']}" for r in results)
        except Exception:
            time.sleep(2)
    return "DuckDuckGo returned no results. Use wikipedia_search, or open a known URL with visit_webpage."


@tool
def ask_youtube(url: str, question: str) -> str:
    """Watches a public YouTube video and answers a question about its visual and audio content (counting things on screen, what is said, etc.).

    Args:
        url: The full YouTube URL.
        question: A specific question about the video.
    """
    try:
        resp = gclient.models.generate_content(
            model=model.model_id,
            contents=types.Content(parts=[
                types.Part(file_data=types.FileData(file_uri=url)),
                types.Part(text=question),
            ]),
        )
        return resp.text or "No answer returned."
    except Exception as e:
        return f"Video analysis failed: {e}"
@tool
def wikipedia_search(query: str) -> str:
    """Searches English Wikipedia and returns matching page titles, URLs, and snippets. Use it to find the right Wikipedia page, then open it with visit_webpage.

    Args:
        query: Keywords to search for on Wikipedia.
    """
    try:
        r = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params={"action": "query", "list": "search", "srsearch": query, "srlimit": 8, "format": "json"},
            headers=UA, timeout=30,
        )
        r.raise_for_status()
        hits = r.json()["query"]["search"]
        if not hits:
            return "No Wikipedia results."
        return "\n\n".join(
            f"{h['title']}\nhttps://en.wikipedia.org/wiki/{quote(h['title'].replace(' ', '_'))}\n{re.sub('<[^>]+>', '', h['snippet'])}"
            for h in hits
        )
    except Exception as e:
        return f"Wikipedia search failed: {e}"


@tool
def visit_webpage(url: str, start: int = 0) -> str:
    """Opens a webpage and returns its content as markdown (tables included), up to 15000 characters at a time. Works for Wikipedia and most sites.

    Args:
        url: The full URL to open.
        start: Character offset to start from. Use it to read further down long pages (for example 15000, then 30000).
    """
    try:
        m = re.match(r"https?://(\w+)\.wikipedia\.org/wiki/([^#?]+)", url)
        if m:
            url = f"https://{m.group(1)}.wikipedia.org/api/rest_v1/page/html/{quote(m.group(2), safe='_()%,')}"
        r = requests.get(url, headers=UA, timeout=30)
        r.raise_for_status()
        text = markdownify(r.text) if "html" in r.headers.get("content-type", "") else r.text
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        chunk = text[start:start + 15000]
        more = f"\n\n[Page continues: call again with start={start + 15000}]" if len(text) > start + 15000 else ""
        return chunk + more
    except Exception as e:
        return f"Error fetching the webpage: {e}"



Aquí creamos una herramienta para que el modelo peuda leer los archivos necesarios para completar las preguntas, y otros para definir la búsqueda web


In [37]:
RULES = """You are solving GAIA benchmark questions. Each has ONE exact, verifiable answer, and grading is an exact string match, so content AND format both matter.

HOW TO WORK
- Read the question twice and note exactly what is asked: which entity, which year, which unit, rounding, ordering, how many items.
- Plan first, then act. Break the question into sub-questions and search for each one separately with short, specific queries. Try several different queries before giving up.
- Open the actual source pages with visit_webpage (Wikipedia, official sites, papers). Search snippets are often incomplete or wrong. Cross-check key facts in a second source.
- If the question mentions an attached file, image, spreadsheet, document, or audio, call read_task_file with the task_id first and base your answer on its contents.
- Do ALL counting, arithmetic, sorting, filtering, and date math in Python. Never do it in your head.
- If the question text looks reversed, encoded, or scrambled, decode it in Python first, then answer what it actually asks.
- If a tool fails or returns nothing, try a different query or source instead of guessing.
- If you truly cannot find the answer, give your single best-supported answer. Never answer 'unknown', 'N/A', or explain that you could not find it.

FINAL ANSWER FORMAT (strict)
- Call final_answer with ONLY the answer value: a number, a few words, or a comma-separated list. No sentence, no explanation, no 'FINAL ANSWER:' prefix, no markdown, no quotes.
- Numbers: plain digits only. No thousands separators, no currency or percent signs, no units, unless the question explicitly asks for them. Use the rounding or unit the question requests.
- Strings: no leading articles (a, an, the), no abbreviations unless asked, no trailing period. Write digits as digits.
- Lists: comma-separated with one space after each comma, in the order the question requests (alphabetical or chronological if it says so).
- Names: give exactly the form requested (full name, last name only, first name only).
- Answer exactly the question asked and nothing more.
- For questions about a YouTube video, call ask_youtube(url, question) with the exact URL and a specific question.
- If web_search fails, use wikipedia_search, then visit_webpage on the best page"""

def check_format(final_answer, agent_memory, **kwargs):
    a = str(final_answer).strip()
    if not a:
        raise ValueError("Empty answer. Provide the answer value.")
    if a.upper().startswith("FINAL ANSWER") or a.lower() in {"unknown", "n/a", "none"}:
        raise ValueError("Give only the answer value, and make a best-supported guess instead of 'unknown'.")
    if len(a) > 120 and "," not in a:
        raise ValueError("Answer is too long. Return only the short exact answer, not a sentence.")
    return True


agent = CodeAgent(
    model=model,
    tools=[web_search(), wikipedia_search, visit_webpage, read_task_file, ask_youtube],
    additional_authorized_imports=["pandas", "numpy", "math", "re", "datetime", "json", "csv", "statistics", "itertools", "collections", "io", "unicodedata"],
    instructions=RULES,
    planning_interval=4,
    max_steps=15,
    final_answer_checks=[check_format],
    code_block_tags="markdown",
    executor_kwargs={"timeout_seconds": 90},
    verbosity_level=1,
)
def clean(answer):
    a = str(answer).strip()
    a = re.sub(r"^\s*FINAL ANSWER:\s*", "", a, flags=re.I)
    a = a.strip().strip('"').strip("'").rstrip(".").strip()
    if re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", a):
        a = a.replace(",", "")
    return a

# quick test of both tools before rerunning anything
print(web_search("Mercedes Sosa discography")[:400])
print(visit_webpage("https://en.wikipedia.org/wiki/Mercedes_Sosa")[:500])

DuckDuckGo returned no results. Use wikipedia_search, or open a known URL with visit_webpage.
Mercedes Sosa

Argentine singer (1935–2009)

Not to be confused with [Mercedes Sola](./Mercedes_Sola "Mercedes Sola").

| Mercedes Sosa | |
| --- | --- |
| Portrait by [Annemarie Heinrich](./Annemarie_Heinrich "Annemarie Heinrich"), 1960s | |
| Born | Haydée Mercedes Sosa  (1935-07-09)9 July 1935  [San Miguel de Tucumán](./San_Miguel_de_Tucumán "San Miguel de Tucumán"), Tucumán, Argentina |
| Died | 4 October 2009(2009-10-04) (aged 74)  [Buenos Aires](./Buenos_Aires "Buenos Aires"), Argentina |



Aquí creamos el agente, con sus herramientas disponibles, sus librerías y el prompt que se va a utilizar como instrucciones generales.

In [36]:

def solve(item, retries=3):
    prompt = item["question"]
    if item.get("file_name"):
        prompt += f"\n\n(This task has an attached file. Use read_task_file with task_id='{item['task_id']}' to read it.)"
    for attempt in range(retries):
        try:
            return clean(agent.run(prompt))
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(20 * (attempt + 1))  # back off on rate limits
    return ""

# Test on the first 3 questions first
for item in questions[:3]:
    print("\nQ:", item["question"][:150])
    print("A:", solve(item))


Q: How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest 2022 version of english wikipedia.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest │
│ 2022 version of english wikipedia.                                                                              │
│                                                                                                                 │
╰─ OpenAIModel - gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- Artist: Mercedes Sosa
- Target period: 2000 to 2009 (inclusive)
- Type of work: Studio albums
- Source requirement: Latest 2022 version of English Wikipedia (or general English Wikipedia as the primary source 
of discography information).

### 1.2. Facts to look up
- Complete discography of Mercedes Sosa, specifically distinguishing studio albums from live albums, compilations, 
etc.
- Release years of Mercedes Sosa's studio albums released between 2000 and 2009.
- Source to find this: English Wikipedia article on Mercedes Sosa or her discography page, accessed via 
`wikipedia_search` and `visit_webpage`.

### 1.3. Facts to derive
- Filter the list of Mercedes Sosa's studio albums to include only those released in the years 2000, 2001, 2002, 
2003, 2004, 2005, 2006, 2007, 2008, and 2009.
- Count the total number of qualifying studio albums.

## 2. Plan
1. Search for Mercedes Sosa's Wikipedia page using `wikipedia_search`.
2. Visit the main Wikipedia page or dedicated discography page of Mercedes Sosa using `visit_webpage`.
3. Extract the list of studio albums and their release dates.
4. Filter the studio albums released strictly between the years 2000 and 2009 inclusive.
5. Count the filtered albums to determine the final total.
6. Submit the final answer using `final_answer`.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = wikipedia_search(query="Mercedes Sosa discography")                                             
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Mercedes Sosa
https://en.wikipedia.org/wiki/Mercedes_Sosa
Haydée Mercedes Sosa (9 July 1935 – 4 October 2009) was an Argentine singer who was popular throughout Latin 
America and many countries outside the region

Cantora, un Viaje Íntimo
https://en.wikipedia.org/wiki/Cantora%2C_un_Viaje_%C3%8Dntimo
Argentine singer Mercedes Sosa, released in 2009 through Sony Music Argentina. The album features Cantora 1 and 
Cantora 2, the project is Sosa&#039;s final album

Joan Baez discography
https://en.wikipedia.org/wiki/Joan_Baez_discography
Three Voices: Live in Concert (1988 concert w/ Konstantin Wecker &amp; Mercedes Sosa), (DVD 2004) How Sweet the 
Sound, American Masters/Razor &amp; Tie (DVD

Liora (singer)
https://en.wikipedia.org/wiki/Liora_%28singer%29
to 2006, she lived in Argentina and performed as a guest singer with Mercedes Sosa and other artists of Latin 
American music. She also does yoga. She lives

José José
https://en.wikipedia.org/wiki/Jos%C3%A9_Jos%C3%A9
José Rómulo Sosa Ortiz (17 February 1948 – 28 September 2019), known professionally as José José (Spanish 
pronunciation: [xoˈse xoˈse]) was a Mexican singer

Sara González Gómez
https://en.wikipedia.org/wiki/Sara_Gonz%C3%A1lez_G%C3%B3mez
 Pablo Milanés, Augusto Blanca, Joan Manuel Serrat, Chico Buarque, Mercedes Sosa, Soledad Bravo, Daniel Viglietti, 
Pete Seeger, Roy Brown, Pedro Guerra

Víctor Heredia
https://en.wikipedia.org/wiki/V%C3%ADctor_Heredia
needed] He has recorded with various artists, such as Joan Manuel Serrat, Mercedes Sosa, León Gieco, Milton 
Nascimento, Cuarteto Zupay, Silvio Rodríguez, and

Holly Near
https://en.wikipedia.org/wiki/Holly_Near
of musicians, including Ronnie Gilbert, Pete Seeger, Arlo Guthrie, Mercedes Sosa, Bernice Johnson Reagon, Bonnie 
Raitt, Jackson Browne, Meg (Shambhavi)

Out: None

... (62 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("right")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: right

[Step 2: Duration 202.55 seconds| Input tokens: 7,019 | Output tokens: 76]

A: right


Aquí vamos haciendo que el agente responda una a una algunas de las las preguntas, con la posibilidad de reintentar ante cualquier error.

In [33]:
answers = []
for i, item in enumerate(questions):
    print(f"\n=== {i+1}/{len(questions)} ===")
    ans = solve(item)
    print("->", ans)
    answers.append({"task_id": item["task_id"], "submitted_answer": ans})
    time.sleep(10)

import json
json.dump(answers, open("answers.json", "w"), indent=2)


=== 1/20 ===


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest │
│ 2022 version of english wikipedia.                                                                              │
│                                                                                                                 │
╰─ OpenAIModel - gemini-3.5-flash-lite ───────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
## 1. Facts survey

### 1.1. Facts given in the task
- Artist name: Mercedes Sosa
- Time period: 2000 to 2009 (inclusive)
- Type of work: Studio albums
- Source to use: The latest 2022 version of English Wikipedia (or English Wikipedia generally regarding her 
discography).

### 1.2. Facts to look up
- Mercedes Sosa's complete discography, specifically distinguishing studio albums from live albums, compilations, 
etc.
- Release dates (years) of each studio album released by Mercedes Sosa.
- Source to find this: English Wikipedia page for Mercedes Sosa or her discography.

### 1.3. Facts to derive
- The count of studio albums released by Mercedes Sosa where the release year falls between 2000 and 2009 
(inclusive), derived by filtering the looked-up discography list by year.

## 2. Plan
1. Search Wikipedia for Mercedes Sosa's discography or main biography page using `wikipedia_search`.
2. Visit the relevant Wikipedia page containing her discography using `visit_webpage`.
3. Extract the list of studio albums and their respective release years.
4. Filter the list to include only studio albums released in the years 2000 through 2009 (inclusive).
5. Count the filtered albums and submit the final count using `final_answer`.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = wikipedia_search(query="Mercedes Sosa discography")                                             
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Mercedes Sosa
https://en.wikipedia.org/wiki/Mercedes_Sosa
Haydée Mercedes Sosa (9 July 1935 – 4 October 2009) was an Argentine singer who was popular throughout Latin 
America and many countries outside the region

Cantora, un Viaje Íntimo
https://en.wikipedia.org/wiki/Cantora%2C_un_Viaje_%C3%8Dntimo
Argentine singer Mercedes Sosa, released in 2009 through Sony Music Argentina. The album features Cantora 1 and 
Cantora 2, the project is Sosa&#039;s final album

Joan Baez discography
https://en.wikipedia.org/wiki/Joan_Baez_discography
Three Voices: Live in Concert (1988 concert w/ Konstantin Wecker &amp; Mercedes Sosa), (DVD 2004) How Sweet the 
Sound, American Masters/Razor &amp; Tie (DVD

Liora (singer)
https://en.wikipedia.org/wiki/Liora_%28singer%29
to 2006, she lived in Argentina and performed as a guest singer with Mercedes Sosa and other artists of Latin 
American music. She also does yoga. She lives

José José
https://en.wikipedia.org/wiki/Jos%C3%A9_Jos%C3%A9
José Rómulo Sosa Ortiz (17 February 1948 – 28 September 2019), known professionally as José José (Spanish 
pronunciation: [xoˈse xoˈse]) was a Mexican singer

Sara González Gómez
https://en.wikipedia.org/wiki/Sara_Gonz%C3%A1lez_G%C3%B3mez
 Pablo Milanés, Augusto Blanca, Joan Manuel Serrat, Chico Buarque, Mercedes Sosa, Soledad Bravo, Daniel Viglietti, 
Pete Seeger, Roy Brown, Pedro Guerra

Víctor Heredia
https://en.wikipedia.org/wiki/V%C3%ADctor_Heredia
needed] He has recorded with various artists, such as Joan Manuel Serrat, Mercedes Sosa, León Gieco, Milton 
Nascimento, Cuarteto Zupay, Silvio Rodríguez, and

Holly Near
https://en.wikipedia.org/wiki/Holly_Near
of musicians, including Ronnie Gilbert, Pete Seeger, Arlo Guthrie, Mercedes Sosa, Bernice Johnson Reagon, Bonnie 
Raitt, Jackson Browne, Meg (Shambhavi)

Out: None

... (915 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Claus")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Claus

[Step 5: Duration 1.08 seconds| Input tokens: 33,887 | Output tokens: 456]

-> Claus


Aquí corremos todas las 20 preguntas y guardamos sus respuestas.

In [34]:
AGENT_CODE = "https://huggingface.co/spaces/LisSanchez-DS/Unit4_GAIA_Agent/tree/main"
payload = {"username": "LisSanchez-DS", "agent_code": AGENT_CODE, "answers": answers}
r = requests.post(f"{API}/submit", json=payload, timeout=60)
print(r.status_code, r.text)

200 {"username":"LisSanchez-DS","score":45.0,"correct_count":9,"total_attempted":20,"message":"Score calculated successfully: 9/20 total questions answered correctly (20 valid tasks attempted). High score updated on leaderboard.","timestamp":"2026-11-15T04:35:54.185423+00:00"}


Al finalizar, enviamos las respuestas para ser evaluadas por GAIA.

NOTA: En este caso el modelo utilizado es uno de Google, no el propuesto por Hugging Face en sus cursos. El código fue creado con ayuda de Claude pero direccionado directamente por la estudiante con respecto a framework, procedimiento, arquitectura, etc.